<a href="https://www.kaggle.com/code/lejuin/asl-landmarks-visualization?scriptVersionId=299069101" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

Uses data from Kaggle competition: https://www.kaggle.com/competitions/asl-fingerspelling

## Landmark visualization
Adapted from https://www.kaggle.com/code/nadigshreekanth/data-visualization-using-mediapipe-apis
& https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/hand_landmarker/python/hand_landmarker.ipynb

In [13]:
!pip install mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 75.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.0 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.


In [14]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

In [15]:
def get_hands(seq_df):
    import mediapipe as mp
    from mediapipe.tasks.python.components.containers.landmark import NormalizedLandmark
    #import numpy as np
    
    mp_hands = mp.tasks.vision.HandLandmarksConnections
    mp_drawing = mp.tasks.vision.drawing_utils
    mp_drawing_styles = mp.tasks.vision.drawing_styles
    
    images = []
    all_hand_landmarks = []
    for seq_idx in range(len(seq_df)):
        x_pose = seq_df.iloc[seq_idx].filter(regex="x_right_hand.*").values
        y_pose = seq_df.iloc[seq_idx].filter(regex="y_right_hand.*").values
        z_pose = seq_df.iloc[seq_idx].filter(regex="z_right_hand.*").values

        right_hand_landmarks = [NormalizedLandmark(x, y, z) for x, y, z in zip(x_pose, y_pose, z_pose)]

        right_hand_image = np.zeros((900, 600, 3), dtype="int8")

        mp_drawing.draw_landmarks(
              right_hand_image,
              right_hand_landmarks,
              mp_hands.HAND_CONNECTIONS,
              mp_drawing_styles.get_default_hand_landmarks_style(),
              mp_drawing_styles.get_default_hand_connections_style())
        
        x_pose = seq_df.iloc[seq_idx].filter(regex="x_left_hand.*").values
        y_pose = seq_df.iloc[seq_idx].filter(regex="y_left_hand.*").values
        z_pose = seq_df.iloc[seq_idx].filter(regex="z_left_hand.*").values

        left_hand_landmarks = [NormalizedLandmark(x, y, z) for x, y, z in zip(x_pose, y_pose, z_pose)]
        
        left_hand_image = np.zeros((900, 600, 3), dtype="int8")

        mp_drawing.draw_landmarks(
                left_hand_image,
                left_hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style())
        
        images.append([right_hand_image.astype(np.uint8), left_hand_image.astype(np.uint8)])
        all_hand_landmarks.append([right_hand_landmarks, left_hand_landmarks])
    return images, all_hand_landmarks

In [16]:
from matplotlib import animation, rc
rc('animation', html='jshtml')

def create_animation(images):
    fig = plt.figure(figsize=(2, 3))
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    im=ax.imshow(images[0], cmap="gray")
    plt.close(fig)
    
    def animate_func(i):
        im.set_array(images[i])
        return [im]

    return animation.FuncAnimation(fig, animate_func, frames=len(images), interval=1000/10)

In [17]:
train = pd.read_csv("/kaggle/input/asl-fingerspelling/train.csv")
train.head(10)

,path,file_id,sequence_id,participant_id,phrase
0,train_landmarks/5414471.parquet,5414471,1816796431,217,3 creekhouse
1,train_landmarks/5414471.parquet,5414471,1816825349,107,scales/kuhaylah
2,train_landmarks/5414471.parquet,5414471,1816909464,1,1383 william lanier
3,train_landmarks/5414471.parquet,5414471,1816967051,63,988 franklin lane
4,train_landmarks/5414471.parquet,5414471,1817123330,89,6920 northeast 661st road
5,train_landmarks/5414471.parquet,5414471,1817141095,38,www.freem.ne.jp
6,train_landmarks/5414471.parquet,5414471,1817169529,70,https://jsi.is/hukuoka
7,train_landmarks/5414471.parquet,5414471,1817171518,202,239613 stolze street
8,train_landmarks/5414471.parquet,5414471,1817195757,136,242-197-6202
9,train_landmarks/5414471.parquet,5414471,1817216847,93,271097 bayshore boulevard


In [18]:
seq_id=1817141095
file_id=train.loc[train['sequence_id']==seq_id, 'file_id'].iloc[0]
path = f"/kaggle/input/asl-fingerspelling/train_landmarks/{file_id}.parquet"

In [20]:
df=pq.read_table(
    path,
    filters=[['sequence_id', '=', seq_id]]
).to_pandas()

In [21]:
# df corresponds to a selected sequence
df.index.unique()

Index([1817141095], dtype='int64', name='sequence_id')

In [22]:
df.shape

(147, 1630)

In [23]:
hand_images, hand_landmarks = get_hands(df)

2026-02-21 07:34:17.700320: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771659257.999763     244 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771659258.081493     244 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771659258.788640     244 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771659258.788688     244 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771659258.788691     244 computation_placer.cc:177] computation placer alr

In [24]:
create_animation(np.array(hand_images)[:, 0])